In [ ]:
%%sql
SELECT * FROM customers LIMIT 5;
SELECT * FROM marketing_channels;
SELECT * FROM categories LIMIT 5;
SELECT * FROM products LIMIT 5;
SELECT * FROM sellers LIMIT 5;
SELECT * FROM orders LIMIT 5;
SELECT * FROM order_items LIMIT 5;

In [ ]:
%%sql
SELECT name, COUNT(*)
FROM customers
GROUP BY name
HAVING COUNT(*) >= 1
LIMIT 10;

In [2]:
%%sql
-- 1. Criar uma tabela grande (pode demorar uns segundos)
CREATE TABLE big_data AS 
SELECT generate_series(1, 500000) AS id, md5(random()::text) AS payload;

-- 2. Configurar work_mem baixa propositalmente (64KB) para a sessão atual
SET work_mem = '64kB';

-- 3. Rodar um Explain Analyze para ver o SORT
-- Procure por "Disk: xxxkB" na saída. Isso significa que a memória não deu e ele usou o disco.
EXPLAIN (ANALYZE, BUFFERS) 
SELECT * FROM big_data ORDER BY payload;

-- 4. Agora aumentar a work_mem para 64MB
SET work_mem = '64MB';

-- 5. Rodar novamente. 
-- Procure por "Method: quicksort  Memory: xxxkB". Agora ele fez tudo na RAM!
EXPLAIN (ANALYZE, BUFFERS) 
SELECT * FROM big_data ORDER BY payload;

 * postgresql://postgres:***@db:5432/lab_dados
500000 rows affected.
Done.
16 rows affected.
Done.
8 rows affected.


QUERY PLAN
Sort (cost=59769.61..61092.63 rows=529209 width=36) (actual time=601.937..652.962 rows=500000 loops=1)
Sort Key: payload
Sort Method: quicksort Memory: 51351kB
Buffers: shared hit=2144 read=2023
-> Seq Scan on big_data (cost=0.00..9459.09 rows=529209 width=36) (actual time=0.026..20.573 rows=500000 loops=1)
Buffers: shared hit=2144 read=2023
Planning Time: 0.034 ms
Execution Time: 664.788 ms


In [4]:
%%sql
-- 1. Pegar a posição atual do WAL (Marcar o início)
SELECT pg_current_wal_lsn() AS inicio_lsn; 
-- Copie o valor que der aqui (Ex: 0/1A2B3C)

-- 2. Fazer uma operação de escrita (Update em tudo)
UPDATE big_data SET payload = md5(id::text)

 * postgresql://postgres:***@db:5432/lab_dados
1 rows affected.
500000 rows affected.


[]

 * postgresql://postgres:***@db:5432/lab_dados
(psycopg2.errors.ActiveSqlTransaction) ALTER SYSTEM cannot run inside a transaction block

[SQL: -- 1. Ativar log de checkpoint (caso não esteja no conf)
ALTER SYSTEM SET log_checkpoints = 'on';]
(Background on this error at: https://sqlalche.me/e/20/2j85)
